# Notebook 09 — Multi-City Hierarchical Analysis

1. Tag cities in real dataset
2. Generate synthetic inter-city corridors
3. Build city super-graph + facility subgraphs
4. City delay ranking and inter-city bottleneck corridors
5. Transfer learning discussion

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from src.data.city_extender import run as build_multicity, CITY_COORDS
from src.graph.hierarchical import (
    build_city_super_graph, build_city_subgraphs,
    city_delay_ranking, intercity_bottleneck_corridors, save_hierarchical
)
print('Multi-city modules loaded')

In [ ]:
# Build multi-city dataset
import os
if not os.path.exists('data/city_augmented/multi_city_trips.parquet'):
    df_mc = build_multicity()
else:
    df_mc = pd.read_parquet('data/city_augmented/multi_city_trips.parquet')

print(f'Multi-city dataset: {df_mc.shape}')
print(f'Real trips: {(~df_mc["is_intercity"]).sum() if "is_intercity" in df_mc.columns else "N/A"}')
print(f'Synthetic inter-city trips: {df_mc["is_intercity"].sum() if "is_intercity" in df_mc.columns else "N/A"}')

In [ ]:
# City super-graph
G_city = build_city_super_graph(df_mc)
print(f'City super-graph: {G_city.number_of_nodes()} cities, {G_city.number_of_edges()} corridors')

ranking = city_delay_ranking(G_city)
print('\nCity delay ranking (worst first):')
print(ranking.head(10).to_string())

In [ ]:
# City delay map
coords_df = pd.DataFrame([(c, lat, lon) for c, (lat, lon) in CITY_COORDS.items()],
                         columns=['city','lat','lon'])
merged = ranking.merge(coords_df, on='city', how='left')

fig = px.scatter_mapbox(
    merged.dropna(subset=['lat']),
    lat='lat', lon='lon',
    size='avg_outgoing_delay_ratio',
    color='avg_outgoing_delay_ratio',
    color_continuous_scale='RdYlGn_r',
    hover_name='city',
    hover_data={'avg_outgoing_delay_ratio': ':.3f', 'chronic_corridors': True},
    mapbox_style='carto-darkmatter',
    zoom=3.5, center={'lat': 22, 'lon': 79},
    title='City Delay Intensity Map',
    size_max=40,
)
fig.update_layout(template='plotly_dark', height=500)
fig.show()
fig.write_html('reports/09_city_delay_map.html')

In [ ]:
# Inter-city bottleneck corridors
bottlenecks = intercity_bottleneck_corridors(G_city, top_n=15)
print('Top inter-city bottleneck corridors:')
print(bottlenecks[['source_city','destination_city','median_delay_ratio','volume','impact_score']].head(10).to_string())

In [ ]:
# Save hierarchical graphs
from src.graph.builder import load_graph
try:
    G_full = load_graph()
    subgraphs = build_city_subgraphs(G_full, df_mc)
    print(f'Intra-city subgraphs built: {len(subgraphs)} cities')
    save_hierarchical(G_city, subgraphs)
except Exception as e:
    print(f'Full graph not yet built: {e}. Run notebook 03 first.')